# Days 2-3: Collection, Cleaning, and EDA

This notebook collects the approved contingency feed, filters it to the latest three months, freezes the cleaned records, and produces interest summaries.

It does not assign sentiment labels or train a model. Those are Day 4 tasks.

## Source decision

The strict GDELT query did not pass the local volume and time-coverage gate. This run therefore uses the documented Google News RSS contingency. We will use this one source only and retain the query, collection time, and filtering rules.

In [ ]:
from pathlib import Path
from urllib.parse import quote, urlparse
import json
import re
import time
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import display

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
FEED_DIR = RAW_DIR / 'google_news_rss'
CLEAN_DIR = PROJECT_DIR / 'data' / 'clean'
FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'
for directory in [RAW_DIR, FEED_DIR, CLEAN_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RSS_QUERIES = [
    'Nigeria food prices',
    'Nigeria food inflation',
    'Nigeria cost of living',
    'Nigeria rice prices',
    'Nigeria food affordability',
    'Nigeria staple food prices',
    'Nigeria purchasing power food',
]
RSS_URLS = [
    'https://news.google.com/rss/search?q=' + quote(query) + '&hl=en-NG&gl=NG&ceid=NG%3Aen'
    for query in RSS_QUERIES
]
MODE = 'replay'
SOURCE_METADATA_PATH = RAW_DIR / 'collection_metadata.json'

if MODE == 'replay':
    if not SOURCE_METADATA_PATH.exists():
        raise FileNotFoundError('Replay metadata is missing: ' + str(SOURCE_METADATA_PATH))
    source_metadata = json.loads(SOURCE_METADATA_PATH.read_text(encoding='utf-8'))
    COLLECTION_TIME = pd.Timestamp(source_metadata['collection_time_utc'])
else:
    source_metadata = {}
    COLLECTION_TIME = pd.Timestamp.now(tz='UTC')

print('Project root:', PROJECT_DIR.resolve())
print('Mode:', MODE)
print('Collection time:', COLLECTION_TIME)

In [ ]:
feed_payloads = []

if MODE == 'replay':
    feed_paths = sorted(FEED_DIR.glob('feed_*.xml'))
    if not feed_paths:
        raise FileNotFoundError('No saved RSS XML feeds found in ' + str(FEED_DIR))
    saved_queries = source_metadata.get('queries', [])
    saved_urls = source_metadata.get('feed_urls', [])
    for index, feed_path in enumerate(feed_paths):
        query = saved_queries[index] if index < len(saved_queries) else feed_path.name
        feed_url = saved_urls[index] if index < len(saved_urls) else ''
        feed_payloads.append((query, feed_url, feed_path.read_bytes()))
    print('Replayed saved feeds:', len(feed_payloads))
else:
    collection_stamp = COLLECTION_TIME.strftime('%Y%m%dT%H%M%SZ')
    for index, (query, url) in enumerate(zip(RSS_QUERIES, RSS_URLS), start=1):
        response = requests.get(
            url,
            headers={'User-Agent': '3mtt-sentiment-tracker/1.0'},
            timeout=30,
        )
        response.raise_for_status()
        raw_xml = response.content
        raw_path = FEED_DIR / f'feed_{index:02d}_{collection_stamp}.xml'
        raw_path.write_bytes(raw_xml)
        feed_payloads.append((query, url, raw_xml))
        print(f'{index}/{len(RSS_QUERIES)}: {query} -> {len(raw_xml)} bytes')
        time.sleep(1)
    print('Live feeds collected:', len(feed_payloads))

In [ ]:
rows = []

for query, feed_url, raw_xml in feed_payloads:
    root = ET.fromstring(raw_xml)
    for item in root.findall('./channel/item'):
        source_node = item.find('source')
        source_label = source_node.text if source_node is not None else ''
        source_url = source_node.attrib.get('url', '') if source_node is not None else ''
        raw_title = item.findtext('title', default='')
        source_suffix = f' - {source_label}'
        headline = raw_title[:-len(source_suffix)] if source_label and raw_title.endswith(source_suffix) else raw_title

        rows.append({
            'query': query,
            'feed_url': feed_url,
            'article_url': item.findtext('link', default=''),
            'headline': headline,
            'headline_raw': raw_title,
            'pub_date_raw': item.findtext('pubDate', default=''),
            'source_label': source_label,
            'source_domain': urlparse(source_url).netloc.lower(),
        })

articles = pd.DataFrame(rows)
articles['headline'] = (
    articles['headline'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
)
articles['headline_raw'] = articles['headline_raw'].fillna('').astype(str)
articles['pub_datetime'] = pd.to_datetime(articles['pub_date_raw'], errors='coerce', utc=True)
articles['collection_time'] = COLLECTION_TIME
feed_items_raw = len(articles)

print('Feed items parsed:', len(articles))
print('Missing headlines:', int(articles['headline'].eq('').sum()))
display(articles.head(5))

In [ ]:
FOOD_TERMS = [
    'food', 'rice', 'bread', 'cooking', 'staple', 'grocery', 'diet',
    'maize', 'yam', 'beans', 'food security', 'food crisis',
]
PRICE_TERMS = [
    'price', 'prices', 'inflation', 'affordability', 'cost',
    'expensive', 'scarcity', 'shortage', 'market', 'hardship',
]
TOPIC_PHRASES = [
    'cost of living', 'purchasing power', 'food inflation',
    'food prices', 'food crisis', 'food affordability',
]

def contains_any(text, terms):
    text = text.lower()
    return any(term in text for term in terms)

articles['headline_key'] = (
    articles['headline'].str.lower().str.replace(r'[^a-z0-9 ]', '', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
)
articles = articles.drop_duplicates(subset=['article_url']).copy()
articles = articles.drop_duplicates(subset=['headline_key', 'source_label', 'pub_datetime']).copy()
unique_records_after_deduplication = len(articles)

articles['has_food_term'] = articles['headline'].map(lambda value: contains_any(value, FOOD_TERMS))
articles['has_price_term'] = articles['headline'].map(lambda value: contains_any(value, PRICE_TERMS))
articles['has_topic_phrase'] = articles['headline'].map(lambda value: contains_any(value, TOPIC_PHRASES))
articles['is_topic_relevant'] = (
    (articles['has_food_term'] & articles['has_price_term'])
    | articles['has_topic_phrase']
)

cutoff = COLLECTION_TIME - pd.DateOffset(months=3)
recent_articles = articles[articles['pub_datetime'] >= cutoff].copy()
filtered_articles = recent_articles[recent_articles['is_topic_relevant']].copy()

print('Rows after URL/headline deduplication:', len(articles))
print('Three-month cutoff:', cutoff)
print('Rows in latest three months:', len(recent_articles))
print('Rows after topic filter:', len(filtered_articles))
print('Rows removed by topic filter:', len(recent_articles) - len(filtered_articles))

In [ ]:
if filtered_articles.empty:
    raise ValueError('The topic filter produced no records. Review the terms before freezing data.')

frozen = filtered_articles.sort_values('pub_datetime').reset_index(drop=True)
frozen_path = CLEAN_DIR / 'articles_frozen.csv'
frozen.to_csv(frozen_path, index=False)

metadata = {
    'source': 'Google News RSS contingency, multi-query collection',
    'queries': RSS_QUERIES,
    'feed_urls': RSS_URLS,
    'collection_time_utc': COLLECTION_TIME.isoformat(),
    'three_month_cutoff_utc': cutoff.isoformat(),
    'feed_items_raw': int(feed_items_raw),
    'unique_records_after_deduplication': int(unique_records_after_deduplication),
    'collection_mode': MODE,
    'feed_count': int(len(feed_payloads)),
    'rows_after_three_month_filter': int(len(recent_articles)),
    'rows_frozen': int(len(frozen)),
    'distinct_source_labels': int(frozen['source_label'].nunique()),
    'food_terms': FOOD_TERMS,
    'price_terms': PRICE_TERMS,
    'topic_phrases': TOPIC_PHRASES,
    'filter_rule': 'food term plus price term, or an approved topic phrase',
}
metadata_path = RAW_DIR / 'collection_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Frozen dataset:', frozen_path)
print('Metadata:', metadata_path)
print('Frozen rows:', len(frozen))

In [ ]:
print('Frozen date range:', frozen['pub_datetime'].min(), 'to', frozen['pub_datetime'].max())
print('Distinct source labels:', frozen['source_label'].nunique())
print('Missing values:')
display(frozen[['headline', 'pub_datetime', 'source_label']].isna().sum().to_frame('missing'))

print('Sample of filtered headlines for manual review:')
display(frozen[['headline', 'pub_datetime', 'source_label']].sample(min(20, len(frozen)), random_state=42))

In [ ]:
frozen['week'] = frozen['pub_datetime'].dt.tz_localize(None).dt.to_period('W').astype(str)
weekly_interest = (
    frozen.groupby('week')
    .agg(
        article_count=('article_url', 'nunique'),
        distinct_source_count=('source_label', 'nunique'),
    )
    .reset_index()
    .sort_values('week')
)
weekly_path = CLEAN_DIR / 'weekly_interest_summary.csv'
weekly_interest.to_csv(weekly_path, index=False)
display(weekly_interest)
print('Weekly summary:', weekly_path)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly_interest['week'], weekly_interest['article_count'], marker='o')
ax.set_title('Weekly Nigerian news attention: food prices and cost of living')
ax.set_xlabel('Week')
ax.set_ylabel('Distinct article count')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
volume_path = FIGURES_DIR / 'weekly_article_volume.png'
fig.savefig(volume_path, dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly_interest['week'], weekly_interest['distinct_source_count'], marker='o', color='darkorange')
ax.set_title('Weekly distinct news sources')
ax.set_xlabel('Week')
ax.set_ylabel('Distinct source count')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
sources_path = FIGURES_DIR / 'weekly_distinct_sources.png'
fig.savefig(sources_path, dpi=150)
plt.show()

print('Figures saved:', volume_path, 'and', sources_path)

## Days 2-3 exit checklist

- Raw RSS response saved with a collection timestamp.
- Collection metadata records the source, query, cutoff, and filters.
- Cleaned records are frozen in `data/clean/articles_frozen.csv`.
- Topic filtering is documented and sample headlines were displayed for review.
- Weekly article volume and distinct-source summaries were created.
- Interest charts were saved.

The next step is manual review of the frozen sample and then sentiment labelling. No sentiment conclusion should be made from these charts yet.